# 04 Generation & Evaluation

`src.generation` RAG 파이프라인을 구동하고 `src.evaluation` 답변 생성 품질을 hit / recall / bert score 기준 평가.



In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

print(f"Project root: {ROOT}")

Project root: d:\AI\projects\finance-terms-rag-chatbot


In [2]:
from src.common.config import get_settings
from src.retrieval.factory import build_retriever
from src.generation.llm import OllamaGenerator
from src.generation.rag_pipeline import RAGPipeline

settings = get_settings()

retriever = build_retriever(mode="dense", k=5)
generator = OllamaGenerator(model=settings.ollama_model, base_url=settings.ollama_base_url)
rag = RAGPipeline(retriever, generator)

d:\AI\projects\finance-terms-rag-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
query = "금융채와 회사채의 차이점을 알려주세요."
result = rag.answer(query)
print(result["answer"])
print("retrieved_ids:", result["retrieved_ids"])

**Answer:**

Financial cassette and company cassette are distinct concepts within the realm of finance, each serving a unique purpose and structure:

1. **Financial Cassette**: This refers to funds provided by financial institutions or companies through financial intermediaries such as banks and other money markets. It is primarily used for facilitating transactions and delivering assets (e.g., cash). Financial cassettes are designed for immediate transactional purposes, often involving tangible goods.

2. **Company Cassette**: In contrast, company cassette involves the investment of funds into corporate entities. This includes periodic coupon payments to investors, such as in bonds or bonds with coupons. Company cassettes typically focus on providing financial instruments and assets related to companies for long-term use rather than immediate transactional purposes.

In summary, while financial cassette is about delivering tangible assets via financial intermediaries
retrieved_ids: ['

In [ ]:
from src.evaluation.pipeline import run_evaluation

df = run_evaluation(
    eval_csv_path=settings.default_eval_csv_path,
    chunk_json_path=settings.default_chunk_json_path,
    output_csv_path=settings.eval_data_dir / "eval_result.csv",
    retrieval_mode="hybrid",
    ollama_model=settings.ollama_model,
    ollama_base_url=settings.ollama_base_url,
    dense_provider="clova",
    dense_model_name="bge-m3",
    dense_collection_name="docs_clova",
    dense_persist_directory=str(settings.chroma_clova_dir),
    k=5,
)

print(df[["hit", "recall", "bert_score_f1"]].mean(numeric_only=True))